In [ ]:
!pip install pandas sqlalchemy pymysql
!pip install paramiko==2.11.0
!pip install sshtunnel


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\ADMIN\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\ADMIN\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\ADMIN\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [1]:
from sshtunnel import SSHTunnelForwarder

with SSHTunnelForwarder(
    ("13.239.79.237", 22),
    ssh_username="ubuntu",
    ssh_pkey="C:/Sagar/Futsal_ML/Futsal-ML-Analysis/Futsal_Match_Prediction_Womens/db_pass/FutsalozApp_ProdNew.pem",
    remote_bind_address=(
        "futsalozdb.ce8imxcnx4zv.ap-southeast-2.rds.amazonaws.com",
        3306
    )
) as tunnel:

    print("SSH Tunnel Connected")
    print("Local Port:", tunnel.local_bind_port)

c:\Sagar\Futsal_ML\Futsal-ML-Analysis\Futsal_Match_Prediction_Womens\.venv\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Sagar\Futsal_ML\Futsal-ML-Analysis\Futsal_Match_Prediction_Womens\.venv\Lib\site-packages\paramiko\transport.py:253: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  "class": algorithms.TripleDES,


SSH Tunnel Connected
Local Port: 61172


In [5]:
import pandas as pd
from sqlalchemy import create_engine, text  # Import 'text' from sqlalchemy
from sshtunnel import SSHTunnelForwarder

SSH_HOST = "13.239.79.237"
SSH_PORT = 22
SSH_USER = "ubuntu"
SSH_KEY = "C:/Sagar/Futsal_ML/Futsal-ML-Analysis/Futsal_Match_Prediction_Womens/db_pass/FutsalozApp_ProdNew.pem"

MYSQL_HOST = "futsalozdb.ce8imxcnx4zv.ap-southeast-2.rds.amazonaws.com"
MYSQL_PORT = 3306
MYSQL_USER = "admin"
MYSQL_PASSWORD = "Dk2poYagiSAUu04wEBtH"
MYSQL_DATABASE = "futsaloz"

with SSHTunnelForwarder(
    (SSH_HOST, SSH_PORT),
    ssh_username=SSH_USER,
    ssh_pkey=SSH_KEY,
    remote_bind_address=(MYSQL_HOST, MYSQL_PORT)
) as tunnel:

    engine = create_engine(
        f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
        f"@127.0.0.1:{tunnel.local_bind_port}/{MYSQL_DATABASE}"
    )

    query = """
    SELECT
    m.*,
    ht.name AS homeTeamName,
    at.name AS awayTeamName,
    c.name AS competitionName,
    s.name AS seasonName
FROM futsaloz.cmp_matches m
LEFT JOIN futsaloz.teams ht
    ON m.homeTeamId = ht.id
LEFT JOIN futsaloz.teams at
    ON m.awayTeamId = at.id
LEFT JOIN futsaloz.competition c
    ON m.cmp_id = c.id
LEFT JOIN futsaloz.season s
    ON c.season_id = s.id
WHERE m.cmp_id IN (
    155, 239, 325, 447, 532,
	633, 724, 879, 945, 1025, 1083
) and is_bye="false";
    """
    
    # 1. Connect using the engine context manager
    # 2. Wrap the query string in sqlalchemy.text()
    # 3. Load the result rows directly into pandas
    with engine.connect() as conn:
        result = conn.execute(text(query))
        matches_raw = pd.DataFrame(result.fetchall(), columns=result.keys())
    
matches_raw


,id,cmp_id,homeTeam,awayTeam,homeTeamId,awayTeamId,startTime,endTime,startDate,startingTime,...,is_bye,note,is_forfeited,is_cancel,homeTeamKits,awayTeamKits,homeTeamName,awayTeamName,competitionName,seasonName
0,42219,155,T3,T6,25,28,2021-04-28 21:00:00,2021-04-28 21:45:00,2021-04-28,21:00:00,...,false,None,false,no,0,0,Northern Phoenix FC Womens,FC Preston Womens,2021 Series Futsal Victoria Women,2021
1,42220,155,T1,T4,30,29,2021-04-28 21:45:00,2021-04-28 22:30:00,2021-04-28,21:45:00,...,false,None,false,no,0,0,Northcote City FC Women,FC Carlton Heart Womens,2021 Series Futsal Victoria Women,2021
2,42221,155,T2,T5,26,31,2021-04-28 22:30:00,2021-04-28 23:15:00,2021-04-28,22:30:00,...,false,None,false,no,0,0,Fitzroy FC Womens,Amazons FC,2021 Series Futsal Victoria Women,2021
3,42222,155,T3,T4,25,29,2021-05-05 21:00:00,2021-05-05 21:45:00,2021-05-05,21:00:00,...,false,None,false,no,0,0,Northern Phoenix FC Womens,FC Carlton Heart Womens,2021 Series Futsal Victoria Women,2021
4,42223,155,T1,T2,30,26,2021-05-05 21:45:00,2021-05-05 22:30:00,2021-05-05,21:45:00,...,false,None,false,no,0,0,Northcote City FC Women,Fitzroy FC Womens,2021 Series Futsal Victoria Women,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,267198,1083,T2,T3,26,3485,2026-11-27 19:40:00,2026-11-27 20:35:00,2026-11-27,19:40:00,...,false,None,false,no,0,0,Fitzroy FC Womens,Hume FC Women,2026.2 SERIES FUTSAL Premiership Women,2026.2
752,267199,1083,T1,T7,29,3487,2026-11-27 21:00:00,2026-11-27 21:55:00,2026-11-27,21:00:00,...,false,None,false,no,0,0,FC Carlton Heart Womens,Victoria Lions Womens,2026.2 SERIES FUTSAL Premiership Women,2026.2
753,267201,1083,T5,T7,3487,1845,2026-12-04 18:15:00,2026-12-04 19:10:00,2026-12-04,18:15:00,...,false,None,false,no,0,0,Victoria Lions Womens,Eastern One Nil FC Womens,2026.2 SERIES FUTSAL Premiership Women,2026.2
754,267202,1083,T1,T6,936,26,2026-12-04 19:40:00,2026-12-04 20:35:00,2026-12-04,19:40:00,...,false,None,false,no,0,0,Melbourne AKU FC Womens,Fitzroy FC Womens,2026.2 SERIES FUTSAL Premiership Women,2026.2


In [6]:
# import pandas as pd
# from sqlalchemy import create_engine
# from sshtunnel import SSHTunnelForwarder

# SSH_HOST = "13.239.79.237"
# SSH_PORT = 22
# SSH_USER = "ubuntu"
# SSH_KEY = "C:/Sagar/Futsal_ML/Futsal-ML-Analysis/Futsal_Match_Prediction_Womens/db_pass/FutsalozApp_ProdNew.pem"

# MYSQL_HOST = "futsalozdb.ce8imxcnx4zv.ap-southeast-2.rds.amazonaws.com"
# MYSQL_PORT = 3306
# MYSQL_USER = "admin"
# MYSQL_PASSWORD = "Dk2poYagiSAUu04wEBtH"
# MYSQL_DATABASE = "futsaloz"

# with SSHTunnelForwarder(
#     (SSH_HOST, SSH_PORT),
#     ssh_username=SSH_USER,
#     ssh_pkey=SSH_KEY,
#     remote_bind_address=(MYSQL_HOST, MYSQL_PORT)
# ) as tunnel:

#     engine = create_engine(
#         f"mysql+pymysql://{MYSQL_USER}:{MYSQL_PASSWORD}"
#         f"@127.0.0.1:{tunnel.local_bind_port}/{MYSQL_DATABASE}"
#     )

# #     query = """
# # SELECT
# #     m.*,
# #     ht.name AS homeTeamName,
# #     at.name AS awayTeamName,
# #     c.name AS competitionName,
# #     s.name AS seasonName

# # FROM futsaloz.cmp_matches m

# # LEFT JOIN futsaloz.teams ht
# #     ON m.homeTeamId = ht.id

# # LEFT JOIN futsaloz.teams at
# #     ON m.awayTeamId = at.id

# # LEFT JOIN futsaloz.competition c
# #     ON m.cmp_id = c.id

# # LEFT JOIN futsaloz.season s
# #     ON c.season_id = s.id
# # """

# #     query = """
# #     SELECT lmsc.*,
# #     u.first_name,
# #     u.last_name,
# #     u.date_of_birth
# # FROM futsaloz.live_match_score_cards lmsc
# # LEFT JOIN futsaloz.users u
# #     ON lmsc.user_id = u.user_id;
# #     """

#     query = """
#     SELECT
#     m.*,
#     ht.name AS homeTeamName,
#     at.name AS awayTeamName,
#     c.name AS competitionName,
#     s.name AS seasonName
# FROM futsaloz.cmp_matches m
# LEFT JOIN futsaloz.teams ht
#     ON m.homeTeamId = ht.id
# LEFT JOIN futsaloz.teams at
#     ON m.awayTeamId = at.id
# LEFT JOIN futsaloz.competition c
#     ON m.cmp_id = c.id
# LEFT JOIN futsaloz.season s
#     ON c.season_id = s.id
# WHERE m.cmp_id IN (
#     155, 239, 325, 447, 532,
# 	633, 724, 879, 945, 1025, 1083
# ) and is_bye="false";
#     """
#     matches_raw = pd.read_sql(query, engine)
    
# matches_raw

In [7]:
matches = matches_raw.copy()
matches

,id,cmp_id,homeTeam,awayTeam,homeTeamId,awayTeamId,startTime,endTime,startDate,startingTime,...,is_bye,note,is_forfeited,is_cancel,homeTeamKits,awayTeamKits,homeTeamName,awayTeamName,competitionName,seasonName
0,42219,155,T3,T6,25,28,2021-04-28 21:00:00,2021-04-28 21:45:00,2021-04-28,21:00:00,...,false,None,false,no,0,0,Northern Phoenix FC Womens,FC Preston Womens,2021 Series Futsal Victoria Women,2021
1,42220,155,T1,T4,30,29,2021-04-28 21:45:00,2021-04-28 22:30:00,2021-04-28,21:45:00,...,false,None,false,no,0,0,Northcote City FC Women,FC Carlton Heart Womens,2021 Series Futsal Victoria Women,2021
2,42221,155,T2,T5,26,31,2021-04-28 22:30:00,2021-04-28 23:15:00,2021-04-28,22:30:00,...,false,None,false,no,0,0,Fitzroy FC Womens,Amazons FC,2021 Series Futsal Victoria Women,2021
3,42222,155,T3,T4,25,29,2021-05-05 21:00:00,2021-05-05 21:45:00,2021-05-05,21:00:00,...,false,None,false,no,0,0,Northern Phoenix FC Womens,FC Carlton Heart Womens,2021 Series Futsal Victoria Women,2021
4,42223,155,T1,T2,30,26,2021-05-05 21:45:00,2021-05-05 22:30:00,2021-05-05,21:45:00,...,false,None,false,no,0,0,Northcote City FC Women,Fitzroy FC Womens,2021 Series Futsal Victoria Women,2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
751,267198,1083,T2,T3,26,3485,2026-11-27 19:40:00,2026-11-27 20:35:00,2026-11-27,19:40:00,...,false,None,false,no,0,0,Fitzroy FC Womens,Hume FC Women,2026.2 SERIES FUTSAL Premiership Women,2026.2
752,267199,1083,T1,T7,29,3487,2026-11-27 21:00:00,2026-11-27 21:55:00,2026-11-27,21:00:00,...,false,None,false,no,0,0,FC Carlton Heart Womens,Victoria Lions Womens,2026.2 SERIES FUTSAL Premiership Women,2026.2
753,267201,1083,T5,T7,3487,1845,2026-12-04 18:15:00,2026-12-04 19:10:00,2026-12-04,18:15:00,...,false,None,false,no,0,0,Victoria Lions Womens,Eastern One Nil FC Womens,2026.2 SERIES FUTSAL Premiership Women,2026.2
754,267202,1083,T1,T6,936,26,2026-12-04 19:40:00,2026-12-04 20:35:00,2026-12-04,19:40:00,...,false,None,false,no,0,0,Melbourne AKU FC Womens,Fitzroy FC Womens,2026.2 SERIES FUTSAL Premiership Women,2026.2


In [8]:
matches.columns

Index(['id', 'cmp_id', 'homeTeam', 'awayTeam', 'homeTeamId', 'awayTeamId',
       'startTime', 'endTime', 'startDate', 'startingTime', 'courtId',
       'courtName', 'quaterFinal', 'semiFinals', 'final', 'referee_id',
       'winningTeam', 'losingTeam', 'winningTeamGoals', 'losingTeamGoals',
       'winningTeamPoints', 'losingTeamPoints', 'status', 'round', 'isDraw',
       'last_match', 'is_bye', 'note', 'is_forfeited', 'is_cancel',
       'homeTeamKits', 'awayTeamKits', 'homeTeamName', 'awayTeamName',
       'competitionName', 'seasonName'],
      dtype='object')

In [9]:
drop_cols = [

    'quaterFinal',
    'semiFinals',
    'final',
    'note',

    'startTime',
    'endTime',
    'startingTime',

    'courtId',
    'courtName',

    'referee_id',

    'homeTeamKits',
    'awayTeamKits',

    'competitionName',

    'last_match'
]

existing = [c for c in drop_cols if c in matches.columns]

matches.drop(columns=existing, inplace=True)

In [10]:
print("Dropped Columns:")
print(existing)

Dropped Columns:
['quaterFinal', 'semiFinals', 'final', 'note', 'startTime', 'endTime', 'startingTime', 'courtId', 'courtName', 'referee_id', 'homeTeamKits', 'awayTeamKits', 'competitionName', 'last_match']


In [11]:
# STEP 5 — CLEAN INVALID MATCHES
matches = matches[
    (matches['is_cancel'] != True) &
    (matches['is_forfeited'] != True)
]

matches = matches[
    matches['status'] == 'Completed'
]

print("Remaining Matches:")
print(matches.shape)

Remaining Matches:
(693, 22)


In [12]:
# STEP 6 — CONVERT DATE & SORT
matches['startDate'] = pd.to_datetime(
    matches['startDate'],
    errors='coerce'
)

matches = matches.sort_values(
    'startDate'
).reset_index(drop=True)

print("Date Converted & Sorted")

print("\nDate Range:")
print(matches['startDate'].min())
print(matches['startDate'].max())

Date Converted & Sorted

Date Range:
2021-04-28 00:00:00
2026-06-26 00:00:00


In [13]:
# STEP 8 — CREATE TARGET VARIABLE
def create_outcome(row):
    if row['winningTeam'] == row['homeTeamId']:
        return 1

    else:
        return 0


matches['outcome'] = matches.apply(
    create_outcome,
    axis=1
)

print("Outcome Distribution:")
print(matches['outcome'].value_counts())

Outcome Distribution:
outcome
1    397
0    296
Name: count, dtype: int64


In [14]:
# STEP 9 — CREATE HOME/AWAY GOALS
import numpy as np

matches['home_goals'] = np.where(

    matches['winningTeam'] ==
    matches['homeTeamId'],

    matches['winningTeamGoals'],
    matches['losingTeamGoals']
)

matches['away_goals'] = np.where(

    matches['winningTeam'] ==
    matches['awayTeamId'],

    matches['winningTeamGoals'],
    matches['losingTeamGoals']
)

display(
    matches[
        [
            'homeTeamName',
            'awayTeamName',
            'home_goals',
            'away_goals'
        ]
    ].head()
)

,homeTeamName,awayTeamName,home_goals,away_goals
0,Northern Phoenix FC Womens,FC Preston Womens,1,5
1,Northcote City FC Women,FC Carlton Heart Womens,3,2
2,Fitzroy FC Womens,Amazons FC,2,4
3,Northern Phoenix FC Womens,FC Carlton Heart Womens,0,6
4,Northcote City FC Women,Fitzroy FC Womens,5,0


In [15]:
# STEP 10 — CREATE FEATURE COLUMNS
feature_cols = [

    # =========================================================
    # STRENGTH FEATURES
    # =========================================================

    'home_win_rate',
    'away_win_rate',

    'home_weighted_form',
    'away_weighted_form',

    'home_attack_strength',
    'away_attack_strength',

    'home_defense_strength',
    'away_defense_strength',

    'home_goal_diff_strength',
    'away_goal_diff_strength',

    # =========================================================
    # STYLE FEATURES
    # =========================================================

    'home_scoring_consistency',
    'away_scoring_consistency',

    'home_clean_sheet_rate',
    'away_clean_sheet_rate',

    'home_high_scoring_rate',
    'away_high_scoring_rate',

    # =========================================================
    # ELO FEATURES
    # =========================================================

    'home_elo',
    'away_elo',

    # =========================================================
    # TACTICAL DIFFERENCES
    # =========================================================

    'diff_form',
    'diff_attack',
    'diff_defense',
    'diff_goal_diff',
    'diff_elo',

    # =========================================================
    # HEAD TO HEAD
    # =========================================================

    'h2h_matches',
    'h2h_home_win_rate',
]

for col in feature_cols:
    matches[col] = 0.0

print("Feature Columns Created")

Feature Columns Created


In [16]:
# STEP 11 — INITIALIZE TEAM MEMORY
INITIAL_ELO = 1500

K_FACTOR = 32

team_memory = {}

print("Team Memory Initialized")

Team Memory Initialized


In [17]:
# STEP 12 — TEAM INITIALIZER FUNCTION
def initialize_team(team_id):

    if team_id not in team_memory:

        team_memory[team_id] = {

            'matches': 0,

            'wins': 0,

            'goals_scored': [],
            'goals_conceded': [],

            'goal_difference': [],

            'results': [],

            'clean_sheets': 0,

            'elo': INITIAL_ELO
        }

print("Initializer Function Ready")

Initializer Function Ready


In [18]:
from collections import defaultdict

# STEP 13 — INITIALIZE H2H MEMORY
h2h_memory = defaultdict(lambda: {
    'matches': 0,
    'team1_wins': 0,
    'team2_wins': 0
})

print("H2H Memory Ready")

H2H Memory Ready


In [19]:
# STEP 14 — MAIN FEATURE ENGINEERING LOOP

for idx, row in matches.iterrows():

    # =========================================================
    # BASIC INFO
    # =========================================================

    home = row['homeTeamId']
    away = row['awayTeamId']

    home_goals = row['home_goals']
    away_goals = row['away_goals']

    outcome = row['outcome']


    # =========================================================
    # INITIALIZE TEAMS
    # =========================================================

    initialize_team(home)
    initialize_team(away)

    hs = team_memory[home]
    aws = team_memory[away]


    # =========================================================
    # WIN RATE
    # =========================================================

    home_wr = (
        hs['wins'] / hs['matches']
        if hs['matches'] > 0 else 0
    )

    away_wr = (
        aws['wins'] / aws['matches']
        if aws['matches'] > 0 else 0
    )

    matches.loc[idx, 'home_win_rate'] = home_wr
    matches.loc[idx, 'away_win_rate'] = away_wr


    # =========================================================
    # WEIGHTED RECENT FORM
    # =========================================================

    def weighted_form(results):
      if len(results) == 0:
        return 0

      last5 = results[-5:]

      weights = np.exp(
        np.linspace(0, 1, len(last5)))

      return np.average(last5, weights=weights)

    home_form = weighted_form(hs['results'])
    away_form = weighted_form(aws['results'])

    matches.loc[idx, 'home_weighted_form'] = home_form
    matches.loc[idx, 'away_weighted_form'] = away_form


    # =========================================================
    # ATTACK STRENGTH
    # =========================================================

    home_attack = (
        np.mean(hs['goals_scored'][-5:])
        if len(hs['goals_scored']) > 0 else 0
    )

    away_attack = (
        np.mean(aws['goals_scored'][-5:])
        if len(aws['goals_scored']) > 0 else 0
    )

    matches.loc[idx, 'home_attack_strength'] = home_attack
    matches.loc[idx, 'away_attack_strength'] = away_attack


    # =========================================================
    # DEFENSE STRENGTH
    # LOWER IS BETTER
    # =========================================================

    home_conceded = (
    np.mean(hs['goals_conceded'][-5:])
    if len(hs['goals_conceded']) > 0 else 0)

    away_conceded = (
    np.mean(aws['goals_conceded'][-5:])
    if len(aws['goals_conceded']) > 0 else 0)

    home_def = 1 / (1 + home_conceded)

    away_def = 1 / (1 + away_conceded)

    matches.loc[idx, 'home_defense_strength'] = home_def
    matches.loc[idx, 'away_defense_strength'] = away_def


    # =========================================================
    # GOAL DIFFERENCE STRENGTH
    # =========================================================

    home_gd = (
        np.mean(hs['goal_difference'][-5:])
        if len(hs['goal_difference']) > 0 else 0
    )

    away_gd = (
        np.mean(aws['goal_difference'][-5:])
        if len(aws['goal_difference']) > 0 else 0
    )

    matches.loc[idx, 'home_goal_diff_strength'] = home_gd
    matches.loc[idx, 'away_goal_diff_strength'] = away_gd


    # =========================================================
    # SCORING CONSISTENCY
    # LOWER STD = MORE CONSISTENT
    # =========================================================

    home_consistency = (
        np.std(hs['goals_scored'][-5:])
        if len(hs['goals_scored']) > 1 else 0
    )

    away_consistency = (
        np.std(aws['goals_scored'][-5:])
        if len(aws['goals_scored']) > 1 else 0
    )

    matches.loc[idx, 'home_scoring_consistency'] = home_consistency
    matches.loc[idx, 'away_scoring_consistency'] = away_consistency


    # =========================================================
    # CLEAN SHEET RATE
    # =========================================================

    home_cs = (
        hs['clean_sheets'] / hs['matches']
        if hs['matches'] > 0 else 0
    )

    away_cs = (
        aws['clean_sheets'] / aws['matches']
        if aws['matches'] > 0 else 0
    )

    matches.loc[idx, 'home_clean_sheet_rate'] = home_cs
    matches.loc[idx, 'away_clean_sheet_rate'] = away_cs


    # =========================================================
    # HIGH SCORING RATE
    # =========================================================

    home_high_scoring = (
        np.mean(
            np.array(hs['goals_scored'][-5:]) >= 4
        )
        if len(hs['goals_scored']) > 0 else 0
    )

    away_high_scoring = (
        np.mean(
            np.array(aws['goals_scored'][-5:]) >= 4
        )
        if len(aws['goals_scored']) > 0 else 0
    )

    matches.loc[idx, 'home_high_scoring_rate'] = home_high_scoring
    matches.loc[idx, 'away_high_scoring_rate'] = away_high_scoring


    # =========================================================
    # ELO
    # =========================================================

    home_elo = hs['elo']
    away_elo = aws['elo']

    matches.loc[idx, 'home_elo'] = home_elo
    matches.loc[idx, 'away_elo'] = away_elo


    # =========================================================
    # DIFFERENTIAL FEATURES
    # =========================================================

    matches.loc[idx, 'diff_form'] = (
        home_form - away_form
    )

    matches.loc[idx, 'diff_attack'] = (
        home_attack - away_attack
    )

    matches.loc[idx, 'diff_defense'] = (
        home_def - away_def
    )

    matches.loc[idx, 'diff_goal_diff'] = (
        home_gd - away_gd
    )

    matches.loc[idx, 'diff_elo'] = (
        home_elo - away_elo
    )


    # =========================================================
    # HEAD TO HEAD
    # =========================================================

    pair = tuple(sorted([home, away]))

    h2h = h2h_memory[pair]

    matches.loc[idx, 'h2h_matches'] = h2h['matches']

    if h2h['matches'] > 0:

      if pair[0] == home:

        home_h2h_winrate = (
            h2h['team1_wins'] / h2h['matches']
        )

      else:

        home_h2h_winrate = (
            h2h['team2_wins'] / h2h['matches']
        )

    else:

      home_h2h_winrate = 0.5

    matches.loc[idx, 'h2h_home_win_rate'] = home_h2h_winrate


    # =========================================================
    # ELO CALCULATION
    # =========================================================

    expected_home = (
        1 / (1 + 10 ** ((away_elo - home_elo)/400))
    )

    if outcome == 1:

        actual_home = 1

    elif outcome == 0:

        actual_home = 0

    else:

        actual_home = 0.5

    new_home_elo = (
        home_elo +
        K_FACTOR * (actual_home - expected_home)
    )

    new_away_elo = (
        away_elo +
        K_FACTOR * ((1-actual_home) - (1-expected_home))
    )


    # =========================================================
    # UPDATE TEAM MEMORY
    # =========================================================

    hs['matches'] += 1
    aws['matches'] += 1

    if outcome == 1:

        hs['wins'] += 1

    elif outcome == 0:

        aws['wins'] += 1


    hs['goals_scored'].append(home_goals)
    hs['goals_conceded'].append(away_goals)

    aws['goals_scored'].append(away_goals)
    aws['goals_conceded'].append(home_goals)


    hs['goal_difference'].append(
        home_goals - away_goals
    )

    aws['goal_difference'].append(
        away_goals - home_goals
    )


    # RESULTS

    if outcome == 1:

        hs['results'].append(1)
        aws['results'].append(0)

    elif outcome == 0:

        hs['results'].append(0)
        aws['results'].append(1)

    else:

        hs['results'].append(0.5)
        aws['results'].append(0.5)


    # CLEAN SHEETS

    if away_goals == 0:
        hs['clean_sheets'] += 1

    if home_goals == 0:
        aws['clean_sheets'] += 1


    # UPDATE ELO

    hs['elo'] = new_home_elo
    aws['elo'] = new_away_elo


    # =========================================================
    # UPDATE H2H
    # =========================================================

    h2h['matches'] += 1

    if outcome == 1:
        h2h['team1_wins'] += 1

    elif outcome == 0:
        h2h['team2_wins'] += 1

print("Advanced Tactical Feature Engineering Complete")

Advanced Tactical Feature Engineering Complete


In [20]:
# =============================================================================
# STEP 15 — CREATE ML DATASET ONLY
# =============================================================================

matches_ml = matches.copy()

remove_cols = [

    # TARGET LEAKAGE
    'winningTeam',
    'losingTeam',

    'winningTeamGoals',
    'losingTeamGoals',

    'winningTeamPoints',
    'losingTeamPoints',

    # IDS
    'match_id',
    'cmp_id',

    # TEXT
    'seasonName',

    # META
    'status',
    'is_bye',
    'is_forfeited',
    'is_cancel'
]

existing = [
    c for c in remove_cols
    if c in matches_ml.columns
]

matches_ml.drop(
    columns=existing,
    inplace=True
)

print("ML Dataset Created")
print(matches_ml.shape)

ML Dataset Created
(693, 38)


In [21]:
# =============================================================================
# CREATE HOME/AWAY GOALS
# =============================================================================

matches['home_goals'] = np.where(
    matches['winningTeam'] == matches['homeTeamId'],
    matches['winningTeamGoals'],
    matches['losingTeamGoals']
)

matches['away_goals'] = np.where(
    matches['winningTeam'] == matches['awayTeamId'],
    matches['winningTeamGoals'],
    matches['losingTeamGoals']
)

print(matches[[
    'homeTeamName',
    'awayTeamName',
    'home_goals',
    'away_goals'
]].head())

                 homeTeamName             awayTeamName  home_goals  away_goals
0  Northern Phoenix FC Womens        FC Preston Womens           1           5
1     Northcote City FC Women  FC Carlton Heart Womens           3           2
2           Fitzroy FC Womens               Amazons FC           2           4
3  Northern Phoenix FC Womens  FC Carlton Heart Womens           0           6
4     Northcote City FC Women        Fitzroy FC Womens           5           0


In [22]:
matches_ml.to_csv(
    "Futsal_match_ml.csv",
    index=False
)

print("Dataset Saved Successfully")

Dataset Saved Successfully


### ML Ready Dataset

In [24]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"C:\Sagar\Futsal_ML\Futsal-ML-Analysis\Futsal_Match_Prediction_Womens\notebooks\Futsal_match_ml.csv"
)

print(df.shape)
df.head()

(693, 38)


,id,homeTeam,awayTeam,homeTeamId,awayTeamId,startDate,round,isDraw,homeTeamName,awayTeamName,...,away_high_scoring_rate,home_elo,away_elo,diff_form,diff_attack,diff_defense,diff_goal_diff,diff_elo,h2h_matches,h2h_home_win_rate
0,42219,T3,T6,25,28,2021-04-28,1,no,Northern Phoenix FC Womens,FC Preston Womens,...,0.0,1500.0,1500.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.5
1,42220,T1,T4,30,29,2021-04-28,1,no,Northcote City FC Women,FC Carlton Heart Womens,...,0.0,1500.0,1500.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.5
2,42221,T2,T5,26,31,2021-04-28,1,no,Fitzroy FC Womens,Amazons FC,...,0.0,1500.0,1500.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.5
3,42222,T3,T4,25,29,2021-05-05,2,no,Northern Phoenix FC Womens,FC Carlton Heart Womens,...,0.0,1484.0,1484.0,0.0,-1.0,-0.083333,-3.0,0.0,0.0,0.5
4,42223,T1,T2,30,26,2021-05-05,2,no,Northcote City FC Women,Fitzroy FC Womens,...,0.0,1516.0,1484.0,1.0,1.0,0.133333,3.0,32.0,0.0,0.5


In [25]:
remove_cols = [
    'isDraw',
    'round'
]

df = df.drop(
    columns=remove_cols,
    errors='ignore'
)

In [26]:
# =============================================================================
# FUNCTION-BASED FUTSAL MATCH PREDICTION PIPELINE
# =============================================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# =============================================================================
# 1. PREPARE DATA
# =============================================================================

def prepare_training_data(feature_df):

    TARGET = 'outcome'

    # DROP_COLS = [
    # 'match_id',
    # 'startDate',

    # 'homeTeamName',
    # 'awayTeamName',

    # 'homeTeamId',
    # 'awayTeamId',

    # 'home_goals',
    # 'away_goals',

    # 'outcome'
    # ]

    # FEATURES = [c for c in feature_df.columns if c not in DROP_COLS]
    # FEATURES = [

    # 'diff_form',
    # 'diff_attack',
    # 'diff_defense',
    # 'diff_goal_diff',
    # 'diff_elo',

    # 'h2h_home_win_rate',

    # 'home_scoring_consistency',
    # 'away_scoring_consistency',

    # 'h2h_matches'
    # ]

    FEATURES = [
    'home_win_rate',
    'away_win_rate',
    'diff_goal_diff',
    'diff_form',
    'diff_attack',
    'diff_defense',
    'diff_elo',
    'h2h_home_win_rate'

    ]

    X = feature_df[FEATURES].copy()
    y = feature_df[TARGET].copy()

    return X, y, FEATURES


# =============================================================================
# 2. TIME-SERIES SPLIT
# =============================================================================

def split_train_test(X, y, split_ratio=0.2, shuffle =True):

    split_index = int(len(X) * (1 - split_ratio))

    X_train = X.iloc[:split_index]
    X_test = X.iloc[split_index:]

    y_train = y.iloc[:split_index]
    y_test = y.iloc[split_index:]

    return X_train, X_test, y_train, y_test


# =============================================================================
# 3. LOGISTIC REGRESSION MODEL
# =============================================================================

def train_logistic_regression(X_train, y_train):

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=1000,
            random_state=42,
        ))
    ])

    model.fit(X_train, y_train)

    return model


# =============================================================================
# 4. RANDOM FOREST MODEL
# =============================================================================

def train_random_forest(X_train, y_train):

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    return model


# =============================================================================
# 5. MODEL EVALUATION
# =============================================================================

def evaluate_model(model, X_test, y_test):

    preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)

    print("=" * 60)
    print("MODEL EVALUATION")
    print("=" * 60)

    print(f"\nAccuracy: {accuracy:.4f}")

    print("\nClassification Report:\n")
    print(classification_report(y_test, preds))

    print("\nConfusion Matrix:\n")
    print(confusion_matrix(y_test, preds))

    return accuracy


# =============================================================================
# 6. FEATURE IMPORTANCE
# =============================================================================

def show_feature_importance(model, feature_names, top_n=15):

    if not hasattr(model, 'feature_importances_'):
        print("Model does not support feature importance")
        return

    import pandas as pd

    fi = pd.DataFrame({
        'feature': feature_names,
        'importance': model.feature_importances_
    })

    fi = fi.sort_values(
        'importance',
        ascending=False
    ).reset_index(drop=True)

    print("=" * 60)
    print("TOP FEATURE IMPORTANCE")
    print("=" * 60)

    for i, row in fi.head(top_n).iterrows():

        bar = "█" * int(row['importance'] * 200)

        print(
            f"{i+1:2d}. "
            f"{row['feature']:<35} "
            f"{row['importance']:.4f} "
            f"{bar}"
        )

    return fi


# =============================================================================
# 7. MATCH INFERENCE / PREDICTION
# =============================================================================

def predict_match(model, feature_row):

    probs = model.predict_proba(feature_row)[0]

    prediction = model.predict(feature_row)[0]

    home_prob = probs[1]
    away_prob = probs[0]

    print("=" * 60)
    print("MATCH PREDICTION")
    print("=" * 60)

    print(f"\nHome Win Probability : {home_prob:.2%}")
    print(f"Away Win Probability : {away_prob:.2%}")

    if prediction == 1:
        print("\nPrediction : HOME WIN")
    else:
        print("\nPrediction : AWAY WIN")

    return {
        'home_win_probability': home_prob,
        'away_win_probability': away_prob,
        'prediction': prediction
    }


# =============================================================================
# 8. EXPLAIN MATCH PREDICTION
# =============================================================================

def explain_match(feature_row):

    print("=" * 60)
    print("TACTICAL MATCH ANALYSIS")
    print("=" * 60)

    row = feature_row.iloc[0]

    print("\n--- FORM ---")

    print(
        f"Form Edge: "
        f"{row['diff_form']:.3f}"
    )

    print("\n--- ATTACK ---")

    print(
        f"Attack Edge: "
        f"{row['diff_attack']:.3f}"
    )

    print("\n--- DEFENSE ---")

    print(
        f"Defense Edge: "
        f"{row['diff_defense']:.3f}"
    )

    print("\n--- GOAL DIFFERENCE ---")

    print(
        f"Goal Difference Edge: "
        f"{row['diff_goal_diff']:.3f}"
    )

    print("\n--- ELO STRENGTH ---")

    print(
        f"ELO Edge: "
        f"{row['diff_elo']:.3f}"
    )

    print("\n--- HEAD TO HEAD ---")

    print(
        f"H2H Home Win Rate: "
        f"{row['h2h_home_win_rate']:.2%}"
    )


# =============================================================================
# 9. COMPLETE PIPELINE RUNNER
# =============================================================================

def run_full_pipeline(feature_df):

    print("=" * 60)
    print("PREPARING DATA")
    print("=" * 60)

    X, y, FEATURES = prepare_training_data(feature_df)
    print("Show:",FEATURES)


    print("Total Features:", len(FEATURES))
    print("Dataset Shape:", X.shape)

    # =========================
    # SPLIT
    # =========================

    X_train, X_test, y_train, y_test = split_train_test(X, y)

    print("\nTrain Shape:", X_train)
    print("Test Shape :", X_test.shape)

    # =========================
    # LOGISTIC REGRESSION
    # =========================

    print("\n")
    print("=" * 60)
    print("TRAINING LOGISTIC REGRESSION")
    print("=" * 60)

    lr_model = train_logistic_regression(
        X_train,
        y_train
    )

    lr_acc = evaluate_model(
        lr_model,
        X_test,
        y_test
    )

    # =========================
    # RANDOM FOREST
    # =========================

    print("\n")
    print("=" * 60)
    print("TRAINING RANDOM FOREST")
    print("=" * 60)

    rf_model = train_random_forest(
        X_train,
        y_train
    )

    rf_acc = evaluate_model(
        rf_model,
        X_test,
        y_test
    )

    # =========================
    # BEST MODEL
    # =========================

    if rf_acc >= lr_acc:

        best_model = rf_model
        best_name = "Random Forest"

    else:

        best_model = lr_model
        best_name = "Logistic Regression"

    print("\n")
    print("=" * 60)
    print(f"BEST MODEL: {best_name}")
    print("=" * 60)

    # =========================
    # FEATURE IMPORTANCE
    # =========================

    if best_name == "Random Forest":

        fi = show_feature_importance(
            best_model,
            FEATURES
        )

    else:

        fi = None

    return {
        'best_model': best_model,
        'best_model_name': best_name,
        'features': FEATURES,
        'X_test': X_test,
        'y_test': y_test,
        'feature_importance': fi
    }


# =============================================================================
# 10. RUN EVERYTHING
# =============================================================================

results = run_full_pipeline(df)

PREPARING DATA
Show: ['home_win_rate', 'away_win_rate', 'diff_goal_diff', 'diff_form', 'diff_attack', 'diff_defense', 'diff_elo', 'h2h_home_win_rate']
Total Features: 8
Dataset Shape: (693, 8)

Train Shape:      home_win_rate  away_win_rate  diff_goal_diff  diff_form  diff_attack  \
0         0.000000       0.000000             0.0   0.000000          0.0   
1         0.000000       0.000000             0.0   0.000000          0.0   
2         0.000000       0.000000             0.0   0.000000          0.0   
3         0.000000       0.000000            -3.0   0.000000         -1.0   
4         1.000000       0.000000             3.0   1.000000          1.0   
..             ...            ...             ...        ...          ...   
549       0.484375       0.401361             5.0   0.260495          3.8   
550       0.398649       0.157407             6.8   0.575927          1.8   
551       0.488372       0.375000             4.8   0.260495          4.0   
552       0.743243     

In [27]:
best_model = results['best_model']

if results['best_model_name'] == "Logistic Regression":
    coef_df = pd.DataFrame({
        'feature': results['features'],
        'coef': best_model.named_steps['model'].coef_[0]
    })
    coef_df = coef_df.sort_values('coef', ascending=False)
elif results['best_model_name'] == "Random Forest":
    coef_df = results['feature_importance']
else:
    coef_df = pd.DataFrame()
    print("Unknown best model type.")

print(coef_df)

             feature  importance
0           diff_elo    0.195792
1     diff_goal_diff    0.186879
2          diff_form    0.150448
3       diff_defense    0.129556
4      home_win_rate    0.107755
5        diff_attack    0.097903
6      away_win_rate    0.084462
7  h2h_home_win_rate    0.047204


In [28]:
matches = pd.read_csv(
    r"C:\Sagar\Futsal_ML\Futsal-ML-Analysis\Futsal_Match_Prediction_Womens\notebooks\Futsal_match_ml.csv"
)
print(matches.shape)

print(matches.columns)

(693, 38)
Index(['id', 'homeTeam', 'awayTeam', 'homeTeamId', 'awayTeamId', 'startDate',
       'round', 'isDraw', 'homeTeamName', 'awayTeamName', 'outcome',
       'home_goals', 'away_goals', 'home_win_rate', 'away_win_rate',
       'home_weighted_form', 'away_weighted_form', 'home_attack_strength',
       'away_attack_strength', 'home_defense_strength',
       'away_defense_strength', 'home_goal_diff_strength',
       'away_goal_diff_strength', 'home_scoring_consistency',
       'away_scoring_consistency', 'home_clean_sheet_rate',
       'away_clean_sheet_rate', 'home_high_scoring_rate',
       'away_high_scoring_rate', 'home_elo', 'away_elo', 'diff_form',
       'diff_attack', 'diff_defense', 'diff_goal_diff', 'diff_elo',
       'h2h_matches', 'h2h_home_win_rate'],
      dtype='object')


In [29]:
# =============================================================================
# BUILD MATCH FEATURE ROW FOR INFERENCE
# =============================================================================

def scoring_consistency(df, team):

    goals = []

    for _, row in df.iterrows():

        if row['homeTeamName'] == team:
            goals.append(row['home_goals'])
        else:
            goals.append(row['away_goals'])

    if len(goals) <= 1:
        return 0

    return np.std(goals)


def build_match_features(
    home_team,
    away_team,
    matches,
    feature_columns
):

    latest_date = matches['startDate'].max()

    # =========================================================
    # HOME TEAM HISTORY
    # =========================================================

    home_hist = matches[
        (matches['homeTeamName'] == home_team) |
        (matches['awayTeamName'] == home_team)
    ].sort_values('startDate')

    # =========================================================
    # AWAY TEAM HISTORY
    # =========================================================

    away_hist = matches[
        (matches['homeTeamName'] == away_team) |
        (matches['awayTeamName'] == away_team)
    ].sort_values('startDate')

    # =========================================================
    # BASIC SAFETY
    # =========================================================

    MIN_MATCHES = 3

    if len(home_hist) < MIN_MATCHES:
      print(f"Limited history for {home_team}")

    if len(away_hist) < MIN_MATCHES:
      print(f"Limited history for {away_team}")

    # =========================================================
    # RECENT FORM
    # =========================================================

    def recent_form(df, team):
      last5 = df.tail(5)
      results = []

      for _, row in last5.iterrows():

        if row['homeTeamName'] == team:

            if row['home_goals'] > row['away_goals']:
              results.append(1)

            elif row['home_goals'] == row['away_goals']:
              results.append(0.5)

            else:
              results.append(0)

        else:

            if row['away_goals'] > row['home_goals']:
                results.append(1)
            else:
                results.append(0)

      return np.mean(results)

    # =========================================================
    # GOALS SCORED / CONCEDED
    # =========================================================

    def attack_strength(df, team):

        goals = []

        for _, row in df.tail(5).iterrows():

            if row['homeTeamName'] == team:
                goals.append(row['home_goals'])

            else:
                goals.append(row['away_goals'])

        return np.mean(goals)

    def defense_strength(df, team):

      conceded = []

      for _, row in df.tail(5).iterrows():

        if row['homeTeamName'] == team:
            conceded.append(row['away_goals'])

        else:
            conceded.append(row['home_goals'])

      avg_conceded = np.mean(conceded)

      # MUST MATCH TRAINING LOGIC
      return 1 / (1 + avg_conceded)

    # =========================================================
    # CALCULATE STATS
    # =========================================================

    home_form = recent_form(home_hist, home_team)
    away_form = recent_form(away_hist, away_team)

    home_attack = attack_strength(home_hist, home_team)
    away_attack = attack_strength(away_hist, away_team)

    home_defense = defense_strength(home_hist, home_team)
    away_defense = defense_strength(away_hist, away_team)

    # =========================================================
    # ELO
    # =========================================================


    matches['home_elo'] = matches['home_elo']
    matches['away_elo'] = matches['away_elo']


    latest_home = home_hist.iloc[-1]

    if latest_home['homeTeamName'] == home_team:
      home_elo = latest_home['home_elo']

    else:
      home_elo = latest_home['away_elo']


    latest_away = away_hist.iloc[-1]

    if latest_away['awayTeamName'] == away_team:
      away_elo = latest_away['away_elo']
    else:
      away_elo = latest_away['home_elo']


    # =========================================================
    # HEAD TO HEAD
    # =========================================================

    h2h = matches[
        (
            (matches['homeTeamName'] == home_team) &
            (matches['awayTeamName'] == away_team)
        ) |
        (
            (matches['homeTeamName'] == away_team) &
            (matches['awayTeamName'] == home_team)
        )
    ]

    h2h_matches = len(h2h)

    home_h2h_wins = 0

    for _, row in h2h.iterrows():
        # Use 'outcome' column instead of 'winningTeam'
        if row['homeTeamName'] == home_team: # 'home_team' from function call was the home team in this H2H match
            if row['outcome'] == 1: # Outcome 1 means the home team won in this match row
                home_h2h_wins += 1
        elif row['awayTeamName'] == home_team: # 'home_team' from function call was the away team in this H2H match
            if row['outcome'] == 0: # Outcome 0 means the away team won in this match row
                home_h2h_wins += 1

    if h2h_matches > 0:
        h2h_rate = home_h2h_wins / h2h_matches
    else:
        h2h_rate = 0.5

    # =========================================================
    # FEATURE ROW
    # =========================================================

    row = {
        'home_win_rate': home_form,
        'away_win_rate': away_form,

        'home_weighted_form': home_form,
        'away_weighted_form': away_form,

        'home_attack_strength': home_attack,
        'away_attack_strength': away_attack,

        'home_defense_strength': home_defense,
        'away_defense_strength': away_defense,

        'home_goal_diff_strength': (
            home_attack - home_defense
        ),

        'away_goal_diff_strength': (
            away_attack - away_defense
        ),

        'home_scoring_consistency': scoring_consistency(home_hist, home_team),
        'away_scoring_consistency': scoring_consistency(away_hist, away_team),

        'home_clean_sheet_rate': (
            1 if home_defense < 2 else 0
        ),

        'away_clean_sheet_rate': (
            1 if away_defense < 2 else 0
        ),

        'home_high_scoring_rate': (
            1 if home_attack > 4 else 0
        ),

        'away_high_scoring_rate': (
            1 if away_attack > 4 else 0
        ),

        'home_elo': home_elo / 100,
        'away_elo': away_elo / 100,

        'diff_form': (
            home_form - away_form
        ),

        'diff_attack': (
            home_attack - away_attack
        ),

        'diff_defense': (
            home_defense - away_defense
        ),

        'diff_goal_diff': (
            (home_attack - home_defense) -
            (away_attack - away_defense)
        ),

        'diff_elo': (
            (home_elo - away_elo) / 100
        ),

        'h2h_matches': h2h_matches,

        'h2h_home_win_rate': h2h_rate
    }

    feature_row = pd.DataFrame([row])

    feature_row = feature_row[feature_columns]

    return feature_row

In [38]:
# =============================================================================
# EXAMPLE MATCH PREDICTION
# =============================================================================

best_model = results['best_model']
feature_columns = results['features']

match_features = build_match_features(
    home_team="Heidelberg United FC Women",
    away_team="FC Preston Womens",
    matches=matches,
    feature_columns=feature_columns
)


match_features

,home_win_rate,away_win_rate,diff_goal_diff,diff_form,diff_attack,diff_defense,diff_elo,h2h_home_win_rate
0,0.4,0.8,-2.728571,-0.4,-2.8,-0.071429,-2.223626,0.333333


In [39]:
# =============================================================================
# PREDICT MATCH
# =============================================================================

prediction = predict_match(
    best_model,
    match_features
)

MATCH PREDICTION

Home Win Probability : 41.68%
Away Win Probability : 58.32%

Prediction : AWAY WIN
